# Initial Data Profiling

## Objective

Profile the public retail sales dataset before data preparation,
forecasting, or business analysis.

The profiling stage will assess:

- File structure
- Dataset size
- Schema
- Data types
- Date coverage
- Missing values
- Duplicate records
- Retail dimensions
- Dataset grain
- Forecasting suitability

No data cleaning or forecasting is performed in this notebook.

In [8]:
from pathlib import Path
import pandas as pd

# Project root
PROJECT_ROOT = Path.cwd().parents[1]

# Raw data directory
RAW_DATA = PROJECT_ROOT / "data" / "raw"

# Display project and data locations
print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DATA)

# Check that the main files exist
for file_name in [
    "train.csv",
    "test.csv",
    "transactions.csv",
    "stores.csv",
    "holidays_events.csv",
    "oil.csv",
]:
    file_path = RAW_DATA / file_name
    print(f"{file_name}: {'FOUND' if file_path.exists() else 'MISSING'}")

Project root: d:\GitHub\retail-sales-forecasting
Raw data: d:\GitHub\retail-sales-forecasting\data\raw
train.csv: FOUND
test.csv: FOUND
transactions.csv: FOUND
stores.csv: FOUND
holidays_events.csv: FOUND
oil.csv: FOUND


## 2.3C — Schema and Data Types

This step inspects the structure of the main public datasets.

We will not modify the source files.

The purpose is to understand the actual schema before designing
data-quality rules, transformations, SQL tables, and forecasting logic.

In [9]:
# Load the main datasets for structural inspection.
# We are only inspecting the schema at this stage.

train_sample = pd.read_csv(
    RAW_DATA / "train.csv",
    nrows=5
)

test_sample = pd.read_csv(
    RAW_DATA / "test.csv",
    nrows=5
)

transactions_sample = pd.read_csv(
    RAW_DATA / "transactions.csv",
    nrows=5
)

stores_sample = pd.read_csv(
    RAW_DATA / "stores.csv",
    nrows=5
)

holidays_sample = pd.read_csv(
    RAW_DATA / "holidays_events.csv",
    nrows=5
)

oil_sample = pd.read_csv(
    RAW_DATA / "oil.csv",
    nrows=5
)

print("TRAIN COLUMNS")
print(train_sample.columns.tolist())

print("\nTRAIN DATA TYPES")
print(train_sample.dtypes)

print("\nTRANSACTIONS COLUMNS")
print(transactions_sample.columns.tolist())

print("\nSTORES COLUMNS")
print(stores_sample.columns.tolist())

print("\nHOLIDAYS COLUMNS")
print(holidays_sample.columns.tolist())

print("\nOIL COLUMNS")
print(oil_sample.columns.tolist())

TRAIN COLUMNS
['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion']

TRAIN DATA TYPES
id               int64
date               str
store_nbr        int64
family             str
sales          float64
onpromotion      int64
dtype: object

TRANSACTIONS COLUMNS
['date', 'store_nbr', 'transactions']

STORES COLUMNS
['store_nbr', 'city', 'state', 'type', 'cluster']

HOLIDAYS COLUMNS
['date', 'type', 'locale', 'locale_name', 'description', 'transferred']

OIL COLUMNS
['date', 'dcoilwtico']


## 2.3D — Dataset Size

Measure the number of records in the main source files.

This establishes the scale of the dataset before deeper
data-quality and time-series analysis.

In [10]:
# Count rows in each source CSV file.
# We use a lightweight chunked approach for the larger files.

def count_csv_rows(file_path, chunk_size=100_000):
    """Count data rows in a CSV without loading the full file into memory."""
    
    row_count = 0
    
    for chunk in pd.read_csv(file_path, chunksize=chunk_size):
        row_count += len(chunk)
    
    return row_count


files_to_count = [
    "train.csv",
    "test.csv",
    "transactions.csv",
    "stores.csv",
    "holidays_events.csv",
    "oil.csv",
]

row_counts = {}

for file_name in files_to_count:
    file_path = RAW_DATA / file_name
    
    row_counts[file_name] = count_csv_rows(file_path)
    
    print(f"{file_name}: {row_counts[file_name]:,} rows")

train.csv: 3,000,888 rows
test.csv: 28,512 rows
transactions.csv: 83,488 rows
stores.csv: 54 rows
holidays_events.csv: 350 rows
oil.csv: 1,218 rows


## 2.3E — Historical Date Coverage

Determine the historical period covered by the training data.

This is required before selecting the forecasting grain and horizon.

We will also inspect the number of unique sales dates,
stores, and product families.

In [11]:
# Read only the columns required for initial coverage profiling.
# We avoid loading unnecessary columns from the 3-million-row dataset.

coverage_df = pd.read_csv(
    RAW_DATA / "train.csv",
    usecols=["date", "store_nbr", "family"]
)

# Convert the date column for profiling only.
coverage_df["date"] = pd.to_datetime(coverage_df["date"])

print("Earliest sales date:", coverage_df["date"].min().date())
print("Latest sales date:", coverage_df["date"].max().date())

print("\nUnique sales dates:", coverage_df["date"].nunique())
print("Unique stores:", coverage_df["store_nbr"].nunique())
print("Unique product families:", coverage_df["family"].nunique())

Earliest sales date: 2013-01-01
Latest sales date: 2017-08-15

Unique sales dates: 1684
Unique stores: 54
Unique product families: 33


## 2.3F — Date Continuity

Check whether the historical sales data contains every calendar date
between the minimum and maximum sales dates.

Missing dates can affect time-series aggregation and forecasting,
so they must be identified before model development.

In [12]:
# Get the unique dates from the training data.
sales_dates = coverage_df["date"].drop_duplicates().sort_values()

# Create the complete calendar date range.
expected_dates = pd.date_range(
    start=sales_dates.min(),
    end=sales_dates.max(),
    freq="D"
)

# Identify dates expected in the calendar but absent from the sales data.
missing_dates = expected_dates.difference(sales_dates)

print("Expected calendar dates:", len(expected_dates))
print("Observed sales dates:", len(sales_dates))
print("Missing calendar dates:", len(missing_dates))

print("\nFirst 20 missing dates:")
print(missing_dates[:20])

Expected calendar dates: 1688
Observed sales dates: 1684
Missing calendar dates: 4

First 20 missing dates:
DatetimeIndex(['2013-12-25', '2014-12-25', '2015-12-25', '2016-12-25'], dtype='datetime64[us]', freq=None)


## 2.3G — Dataset Grain Validation

Determine the lowest logical level represented by the sales dataset.

Initial hypothesis:

> One record per Date + Store + Product Family

We will test whether this combination uniquely identifies
each sales record.

In [13]:
# Load only the columns required to test the dataset grain.
grain_df = pd.read_csv(
    RAW_DATA / "train.csv",
    usecols=["date", "store_nbr", "family", "id"]
)

# Count duplicate combinations of the proposed grain.
duplicate_grain_rows = grain_df.duplicated(
    subset=["date", "store_nbr", "family"]
).sum()

# Count unique combinations of the proposed grain.
unique_grain_combinations = grain_df[
    ["date", "store_nbr", "family"]
].drop_duplicates().shape[0]

# Total records.
total_records = len(grain_df)

print("Total records:", f"{total_records:,}")
print(
    "Unique Date + Store + Family combinations:",
    f"{unique_grain_combinations:,}"
)
print(
    "Duplicate Date + Store + Family records:",
    f"{duplicate_grain_rows:,}"
)

Total records: 3,000,888
Unique Date + Store + Family combinations: 3,000,888
Duplicate Date + Store + Family records: 0


## 2.3H — Record Identifier Validation

Assess whether the `id` column uniquely identifies each sales record.

This helps distinguish a technical row identifier from
the actual business grain of the dataset.

In [14]:
# Validate the uniqueness and range of the source record ID.

id_count = grain_df["id"].count()
unique_id_count = grain_df["id"].nunique()
duplicate_id_count = grain_df["id"].duplicated().sum()

print("Total ID values:", f"{id_count:,}")
print("Unique ID values:", f"{unique_id_count:,}")
print("Duplicate IDs:", f"{duplicate_id_count:,}")

print("\nMinimum ID:", grain_df["id"].min())
print("Maximum ID:", grain_df["id"].max())

Total ID values: 3,000,888
Unique ID values: 3,000,888
Duplicate IDs: 0

Minimum ID: 0
Maximum ID: 3000887


## 2.3I — Missing Value Profiling

Measure missing values in the main sales dataset.

For each column, calculate:

- Total missing values
- Missing percentage

No missing values will be imputed or removed at this stage.
This is profiling only.

In [15]:
# Load the main sales dataset.
# At this stage we need all five analytical columns.

sales_df = pd.read_csv(
    RAW_DATA / "train.csv",
    usecols=[
        "id",
        "date",
        "store_nbr",
        "family",
        "sales",
        "onpromotion"
    ]
)

# Calculate missing-value counts.
missing_counts = sales_df.isna().sum()

# Calculate missing-value percentages.
missing_percentages = (
    sales_df.isna().mean() * 100
)

# Combine the results into a profiling table.
missing_profile = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_percentage": missing_percentages
})

print(missing_profile)

             missing_count  missing_percentage
id                       0                 0.0
date                     0                 0.0
store_nbr                0                 0.0
family                   0                 0.0
sales                    0                 0.0
onpromotion              0                 0.0


## 2.3J — Sales Value Validation

Profile the sales measure to identify:

- Distribution statistics
- Zero-sales records
- Negative-sales records

Zero sales are not automatically considered errors.
Negative sales require investigation before any treatment decision.

In [16]:
# Basic statistical profile of the sales measure.

sales_summary = sales_df["sales"].describe()

zero_sales_count = (sales_df["sales"] == 0).sum()
negative_sales_count = (sales_df["sales"] < 0).sum()

print("Sales summary:")
print(sales_summary)

print("\nZero-sales records:", f"{zero_sales_count:,}")
print("Negative-sales records:", f"{negative_sales_count:,}")

Sales summary:
count    3.000888e+06
mean     3.577757e+02
std      1.101998e+03
min      0.000000e+00
25%      0.000000e+00
50%      1.100000e+01
75%      1.958473e+02
max      1.247170e+05
Name: sales, dtype: float64

Zero-sales records: 939,130
Negative-sales records: 0


## 2.3K — Promotion Field Validation

Profile the `onpromotion` field to understand how promotional
activity is represented in the sales dataset.

We will determine whether the field behaves like a count,
rather than assuming it is a simple yes/no flag.

In [17]:
# Profile the onpromotion field.

promotion_summary = sales_df["onpromotion"].describe()

zero_promotion_count = (
    sales_df["onpromotion"] == 0
).sum()

positive_promotion_count = (
    sales_df["onpromotion"] > 0
).sum()

negative_promotion_count = (
    sales_df["onpromotion"] < 0
).sum()

print("Promotion summary:")
print(promotion_summary)

print("\nZero-promotion records:", f"{zero_promotion_count:,}")
print("Positive-promotion records:", f"{positive_promotion_count:,}")
print("Negative-promotion records:", f"{negative_promotion_count:,}")

Promotion summary:
count    3.000888e+06
mean     2.602770e+00
std      1.221888e+01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      7.410000e+02
Name: onpromotion, dtype: float64

Zero-promotion records: 2,389,559
Positive-promotion records: 611,329
Negative-promotion records: 0


## 2.3L — Supporting Dataset: Stores

Validate the store master data before joining it to the sales dataset.

Checks:
- columns and data types
- row count
- missing values
- duplicate store IDs
- store ID coverage
- store type distribution

In [18]:
# Load and profile the store master data.

stores_df = pd.read_csv(
    RAW_DATA / "stores.csv"
)

print("Stores shape:", stores_df.shape)

print("\nColumns:")
print(stores_df.columns.tolist())

print("\nData types:")
print(stores_df.dtypes)

print("\nMissing values:")
print(stores_df.isna().sum())

print("\nDuplicate store IDs:")
print(stores_df["store_nbr"].duplicated().sum())

print("\nStore ID range:")
print("Minimum:", stores_df["store_nbr"].min())
print("Maximum:", stores_df["store_nbr"].max())

print("\nStore types:")
print(stores_df["type"].value_counts().sort_index())

print("\nFirst 5 rows:")
display(stores_df.head())

Stores shape: (54, 5)

Columns:
['store_nbr', 'city', 'state', 'type', 'cluster']

Data types:
store_nbr    int64
city           str
state          str
type           str
cluster      int64
dtype: object

Missing values:
store_nbr    0
city         0
state        0
type         0
cluster      0
dtype: int64

Duplicate store IDs:
0

Store ID range:
Minimum: 1
Maximum: 54

Store types:
type
A     9
B     8
C    15
D    18
E     4
Name: count, dtype: int64

First 5 rows:


,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8
3,4,Quito,Pichincha,D,9
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4


## 2.3M — Store Referential Integrity

Verify that every `store_nbr` appearing in the sales dataset
exists in the store master.

This prevents orphan sales records when the datasets are joined.

In [19]:
# Validate that every sales store exists in the store master.

sales_store_ids = set(sales_df["store_nbr"].unique())
master_store_ids = set(stores_df["store_nbr"].unique())

missing_from_master = sales_store_ids - master_store_ids
unused_master_stores = master_store_ids - sales_store_ids

print("Stores in sales data:", len(sales_store_ids))
print("Stores in store master:", len(master_store_ids))

print("\nSales stores missing from master:", len(missing_from_master))
print("Missing store IDs:", sorted(missing_from_master))

print("\nMaster stores not present in sales:", len(unused_master_stores))
print("Unused master store IDs:", sorted(unused_master_stores))

Stores in sales data: 54
Stores in store master: 54

Sales stores missing from master: 0
Missing store IDs: []

Master stores not present in sales: 0
Unused master store IDs: []


## 2.3N — Supporting Dataset: Transactions

Profile the transaction dataset before using it as a supporting
retail KPI or forecasting driver.

Checks:
- columns and data types
- row count
- missing values
- duplicate Date × Store records
- date coverage
- store coverage
- negative transactions
- zero transactions

In [20]:
# Load and profile the transaction data.

transactions_df = pd.read_csv(
    RAW_DATA / "transactions.csv"
)

print("Transactions shape:", transactions_df.shape)

print("\nColumns:")
print(transactions_df.columns.tolist())

print("\nData types:")
print(transactions_df.dtypes)

print("\nMissing values:")
print(transactions_df.isna().sum())

print("\nDuplicate Date × Store records:")
print(
    transactions_df.duplicated(
        subset=["date", "store_nbr"]
    ).sum()
)

print("\nDate range:")
print("Earliest:", transactions_df["date"].min())
print("Latest:", transactions_df["date"].max())

print("\nUnique dates:", transactions_df["date"].nunique())
print("Unique stores:", transactions_df["store_nbr"].nunique())

print("\nTransaction summary:")
print(transactions_df["transactions"].describe())

print(
    "\nZero-transaction records:",
    (transactions_df["transactions"] == 0).sum()
)

print(
    "Negative-transaction records:",
    (transactions_df["transactions"] < 0).sum()
)

Transactions shape: (83488, 3)

Columns:
['date', 'store_nbr', 'transactions']

Data types:
date              str
store_nbr       int64
transactions    int64
dtype: object

Missing values:
date            0
store_nbr       0
transactions    0
dtype: int64

Duplicate Date × Store records:
0

Date range:
Earliest: 2013-01-01
Latest: 2017-08-15

Unique dates: 1682
Unique stores: 54

Transaction summary:
count    83488.000000
mean      1694.602158
std        963.286644
min          5.000000
25%       1046.000000
50%       1393.000000
75%       2079.000000
max       8359.000000
Name: transactions, dtype: float64

Zero-transaction records: 0
Negative-transaction records: 0


## 2.3O — Transaction and Sales Date Coverage

Compare the unique dates in the sales and transaction datasets.

The purpose is to identify coverage mismatches before the
datasets are joined for retail KPI or forecasting analysis.

In [21]:
# Compare sales and transaction date coverage.

sales_dates = set(
    pd.to_datetime(sales_df["date"]).dt.date
)

transaction_dates = set(
    pd.to_datetime(transactions_df["date"]).dt.date
)

sales_missing_transactions = (
    sales_dates - transaction_dates
)

transactions_missing_sales = (
    transaction_dates - sales_dates
)

print(
    "Sales dates:",
    len(sales_dates)
)

print(
    "Transaction dates:",
    len(transaction_dates)
)

print(
    "\nSales dates missing from transactions:",
    len(sales_missing_transactions)
)

print(
    sorted(sales_missing_transactions)
)

print(
    "\nTransaction dates missing from sales:",
    len(transactions_missing_sales)
)

print(
    sorted(transactions_missing_sales)
)

Sales dates: 1684
Transaction dates: 1682

Sales dates missing from transactions: 2
[datetime.date(2016, 1, 1), datetime.date(2016, 1, 3)]

Transaction dates missing from sales: 0
[]


## 2.3P — Supporting Dataset: Holidays and Events

Profile the holiday/event calendar to understand special dates
that may explain sales and transaction coverage patterns.

Checks:
- columns and data types
- row count
- missing values
- duplicate date/event records
- date coverage
- holiday types
- transferred events

In [22]:
# Load and profile the holiday/event data.

holidays_df = pd.read_csv(
    RAW_DATA / "holidays_events.csv"
)

print("Holidays shape:", holidays_df.shape)

print("\nColumns:")
print(holidays_df.columns.tolist())

print("\nData types:")
print(holidays_df.dtypes)

print("\nMissing values:")
print(holidays_df.isna().sum())

print("\nDate range:")
print("Earliest:", holidays_df["date"].min())
print("Latest:", holidays_df["date"].max())

print("\nUnique dates:", holidays_df["date"].nunique())

print("\nDuplicate full records:")
print(holidays_df.duplicated().sum())

print("\nEvent types:")
print(holidays_df["type"].value_counts().sort_index())

print("\nTransferred events:")
print(holidays_df["transferred"].value_counts(dropna=False))

print("\nFirst 10 rows:")
display(holidays_df.head(10))

Holidays shape: (350, 6)

Columns:
['date', 'type', 'locale', 'locale_name', 'description', 'transferred']

Data types:
date            str
type            str
locale          str
locale_name     str
description     str
transferred    bool
dtype: object

Missing values:
date           0
type           0
locale         0
locale_name    0
description    0
transferred    0
dtype: int64

Date range:
Earliest: 2012-03-02
Latest: 2017-12-26

Unique dates: 312

Duplicate full records:
0

Event types:
type
Additional     51
Bridge          5
Event          56
Holiday       221
Transfer       12
Work Day        5
Name: count, dtype: int64

Transferred events:
transferred
False    338
True      12
Name: count, dtype: int64

First 10 rows:


,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False
5,2012-05-12,Holiday,Local,Puyo,Cantonizacion del Puyo,False
6,2012-06-23,Holiday,Local,Guaranda,Cantonizacion de Guaranda,False
7,2012-06-25,Holiday,Regional,Imbabura,Provincializacion de Imbabura,False
8,2012-06-25,Holiday,Local,Latacunga,Cantonizacion de Latacunga,False
9,2012-06-25,Holiday,Local,Machala,Fundacion de Machala,False


## 2.3Q — Investigate Coverage Anomalies Using the Holiday Calendar

Inspect holiday/event records for dates identified during
sales and transaction coverage validation.

Focus dates:
- December 25, 2013
- December 25, 2014
- December 25, 2015
- December 25, 2016
- January 1, 2016
- January 3, 2016

In [23]:
# Investigate the holiday/event context for the dates
# identified during data-quality profiling.

investigation_dates = [
    "2013-12-25",
    "2014-12-25",
    "2015-12-25",
    "2016-12-25",
    "2016-01-01",
    "2016-01-03",
]

holiday_check = holidays_df[
    holidays_df["date"].isin(investigation_dates)
].sort_values("date")

display(holiday_check)

,date,type,locale,locale_name,description,transferred
89,2013-12-25,Holiday,National,Ecuador,Navidad,False
155,2014-12-25,Holiday,National,Ecuador,Navidad,False
208,2015-12-25,Holiday,National,Ecuador,Navidad,False
211,2016-01-01,Holiday,National,Ecuador,Primer dia del ano,False
294,2016-12-25,Holiday,National,Ecuador,Navidad,False


## 2.3R — Supporting Dataset: Oil Prices

Profile the oil-price dataset before considering it as an
external business/economic driver for retail sales analysis.

Checks:
- columns and data types
- row count
- missing values
- duplicate dates
- date coverage
- zero and negative prices
- price summary

In [24]:
# Load and profile the oil-price data.

oil_df = pd.read_csv(
    RAW_DATA / "oil.csv"
)

print("Oil shape:", oil_df.shape)

print("\nColumns:")
print(oil_df.columns.tolist())

print("\nData types:")
print(oil_df.dtypes)

print("\nMissing values:")
print(oil_df.isna().sum())

print("\nDuplicate dates:")
print(oil_df["date"].duplicated().sum())

print("\nDate range:")
print("Earliest:", oil_df["date"].min())
print("Latest:", oil_df["date"].max())

print("\nUnique dates:", oil_df["date"].nunique())

print("\nOil price summary:")
print(oil_df["dcoilwtico"].describe())

print(
    "\nZero-price records:",
    (oil_df["dcoilwtico"] == 0).sum()
)

print(
    "Negative-price records:",
    (oil_df["dcoilwtico"] < 0).sum()
)

print("\nFirst 10 rows:")
display(oil_df.head(10))

Oil shape: (1218, 2)

Columns:
['date', 'dcoilwtico']

Data types:
date              str
dcoilwtico    float64
dtype: object

Missing values:
date           0
dcoilwtico    43
dtype: int64

Duplicate dates:
0

Date range:
Earliest: 2013-01-01
Latest: 2017-08-31

Unique dates: 1218

Oil price summary:
count    1175.000000
mean       67.714366
std        25.630476
min        26.190000
25%        46.405000
50%        53.190000
75%        95.660000
max       110.620000
Name: dcoilwtico, dtype: float64

Zero-price records: 0
Negative-price records: 0

First 10 rows:


,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20
5,2013-01-08,93.21
6,2013-01-09,93.08
7,2013-01-10,93.81
8,2013-01-11,93.60
9,2013-01-14,94.27


## 2.3S — Investigate Missing Oil Prices

Identify the dates where the oil-price value is missing.

The purpose is to determine whether the missing values are
associated with non-trading days or require additional treatment.

In [25]:
# Identify dates with missing oil prices.

missing_oil_dates = oil_df[
    oil_df["dcoilwtico"].isna()
].copy()

print(
    "Number of missing oil-price dates:",
    len(missing_oil_dates)
)

print("\nMissing oil-price dates:")
display(missing_oil_dates)

Number of missing oil-price dates: 43

Missing oil-price dates:


,date,dcoilwtico
0,2013-01-01,NaN
14,2013-01-21,NaN
34,2013-02-18,NaN
63,2013-03-29,NaN
104,2013-05-27,NaN
132,2013-07-04,NaN
174,2013-09-02,NaN
237,2013-11-28,NaN
256,2013-12-25,NaN
261,2014-01-01,NaN


## 2.3T — Validate Missing Oil Dates

Classify missing oil-price dates by day of week.

This helps distinguish expected non-trading-day gaps from
unexpected missing observations.

In [26]:
# Classify missing oil-price dates by day of week.

missing_oil_analysis = missing_oil_dates.copy()

missing_oil_analysis["date"] = pd.to_datetime(
    missing_oil_analysis["date"]
)

missing_oil_analysis["day_of_week"] = (
    missing_oil_analysis["date"]
    .dt.day_name()
)

missing_oil_analysis["day_number"] = (
    missing_oil_analysis["date"]
    .dt.dayofweek
)

print("Missing oil prices by day of week:")
print(
    missing_oil_analysis["day_of_week"]
    .value_counts()
    .sort_index()
)

print("\nTotal missing oil-price dates:")
print(len(missing_oil_analysis))

Missing oil prices by day of week:
day_of_week
Friday        9
Monday       23
Thursday      7
Tuesday       2
Wednesday     2
Name: count, dtype: int64

Total missing oil-price dates:
43


## 2.3U — Transaction Store Referential Integrity

Verify that every store appearing in the transaction dataset
exists in the store master.

In [27]:
# Validate transaction store IDs against the store master.

transaction_store_ids = set(
    transactions_df["store_nbr"].unique()
)

missing_transaction_stores = (
    transaction_store_ids - master_store_ids
)

unused_transaction_stores = (
    master_store_ids - transaction_store_ids
)

print(
    "Stores in transactions:",
    len(transaction_store_ids)
)

print(
    "Stores in store master:",
    len(master_store_ids)
)

print(
    "\nTransaction stores missing from master:",
    len(missing_transaction_stores)
)

print(
    "Missing store IDs:",
    sorted(missing_transaction_stores)
)

print(
    "\nMaster stores not present in transactions:",
    len(unused_transaction_stores)
)

print(
    "Unused store IDs:",
    sorted(unused_transaction_stores)
)

Stores in transactions: 54
Stores in store master: 54

Transaction stores missing from master: 0
Missing store IDs: []

Master stores not present in transactions: 0
Unused store IDs: []


# Phase 2(B) — Overall Data Quality Assessment

## Objective

Evaluate whether the source datasets are sufficiently reliable for
data preparation and retail sales forecasting.

The assessment uses the following data-quality dimensions:

1. Completeness
2. Validity
3. Uniqueness
4. Consistency
5. Referential Integrity
6. Temporal Integrity
7. Business Rule Integrity
8. Cross-Dataset Integrity

Each check will be classified as:

- **PASS** — No issue identified based on the tested rule.
- **REVIEW** — An exception exists and requires documented treatment.
- **FAIL** — A material data-quality problem prevents reliable downstream use.

A PASS does not mean the dataset is perfect. It means the specific
tested rule passed.

## Overall Data Quality Assessment

| Dimension | Check | Result | Status | Treatment / Comment |
|---|---|---|---|---|
| Completeness | Missing values in sales dataset | 0 | PASS | No missing values identified |
| Completeness | Missing values in supporting datasets | Identified in oil | REVIEW | Handle during preparation |
| Validity | Negative sales | 0 | PASS | No negative sales identified |
| Validity | Negative promotion counts | 0 | PASS | No negative promotion counts |
| Validity | Negative transactions | 0 | PASS | No negative transaction counts |
| Uniqueness | Duplicate sales grain | 0 | PASS | Date × Store × Family is unique |
| Uniqueness | Duplicate sales IDs | 0 | PASS | Source IDs are unique |
| Uniqueness | Duplicate transaction grain | 0 | PASS | Date × Store is unique |
| Referential Integrity | Sales → Store master | 0 missing | PASS | All sales stores exist in master |
| Referential Integrity | Transactions → Store master | 0 missing | PASS | All transaction stores exist in master |
| Temporal Integrity | Sales calendar continuity | 4 missing dates | REVIEW | Missing dates require documented treatment |
| Cross-Dataset Integrity | Sales vs Transactions dates | 2 sales dates missing | REVIEW | Investigate before joining |
| Cross-Dataset Integrity | Holiday date multiplicity | Multiple records on some dates | REVIEW | Controlled aggregation/join required |
| Business Rule | onpromotion values | Non-negative count | PASS | Treat as promotion count |
| Business Rule | Oil price values | 43 missing | REVIEW | Treatment required if used as feature |
| Business Rule | Sales grain | Confirmed | PASS | Date × Store × Product Family |

In [1]:
# ============================================================
# Step 2B.3 — Overall Data Quality Scorecard
# ============================================================

# Define the status of each assessed data-quality check.
quality_checks = [
    ("Completeness", "Missing values in sales dataset", "PASS"),
    ("Completeness", "Missing values in supporting datasets", "REVIEW"),
    ("Validity", "Negative sales", "PASS"),
    ("Validity", "Negative promotion counts", "PASS"),
    ("Validity", "Negative transactions", "PASS"),
    ("Uniqueness", "Duplicate sales grain", "PASS"),
    ("Uniqueness", "Duplicate sales IDs", "PASS"),
    ("Uniqueness", "Duplicate transaction grain", "PASS"),
    ("Referential Integrity", "Sales → Store master", "PASS"),
    ("Referential Integrity", "Transactions → Store master", "PASS"),
    ("Temporal Integrity", "Sales calendar continuity", "REVIEW"),
    ("Cross-Dataset Integrity", "Sales vs Transactions dates", "REVIEW"),
    ("Cross-Dataset Integrity", "Holiday date multiplicity", "REVIEW"),
    ("Business Rule", "onpromotion values", "PASS"),
    ("Business Rule", "Oil price values", "REVIEW"),
    ("Business Rule", "Sales grain", "PASS"),
]

# Count each status.
status_counts = {}

for _, _, status in quality_checks:
    status_counts[status] = status_counts.get(status, 0) + 1

# Display the results.
print("Overall Data Quality Scorecard")
print("=" * 40)

print("Total checks:", len(quality_checks))
print("PASS:", status_counts.get("PASS", 0))
print("REVIEW:", status_counts.get("REVIEW", 0))
print("FAIL:", status_counts.get("FAIL", 0))

Overall Data Quality Scorecard
Total checks: 16
PASS: 11
REVIEW: 5
FAIL: 0


In [2]:
# Display every quality check and its status.

for dimension, check, status in quality_checks:
    print(f"[{status}] {dimension} — {check}")

[PASS] Completeness — Missing values in sales dataset
[REVIEW] Completeness — Missing values in supporting datasets
[PASS] Validity — Negative sales
[PASS] Validity — Negative promotion counts
[PASS] Validity — Negative transactions
[PASS] Uniqueness — Duplicate sales grain
[PASS] Uniqueness — Duplicate sales IDs
[PASS] Uniqueness — Duplicate transaction grain
[PASS] Referential Integrity — Sales → Store master
[PASS] Referential Integrity — Transactions → Store master
[REVIEW] Temporal Integrity — Sales calendar continuity
[REVIEW] Cross-Dataset Integrity — Sales vs Transactions dates
[REVIEW] Cross-Dataset Integrity — Holiday date multiplicity
[PASS] Business Rule — onpromotion values
[REVIEW] Business Rule — Oil price values
[PASS] Business Rule — Sales grain


## Data Quality Exceptions Requiring Review

The initial assessment identified five REVIEW areas. These are not
automatically treated as data errors; they require documented treatment
before downstream analysis or forecasting.

### 1. Missing values in supporting datasets

The oil dataset contains **43 missing price observations**.

Treatment decision:
- Preserve the original raw oil data.
- Investigate the missing-date pattern during data preparation.
- Define an appropriate treatment only if oil price is used as a forecasting
  feature.
- Do not modify the raw source file.

### 2. Sales calendar continuity

The sales dataset contains **4 missing calendar dates**:

- 2013-12-25
- 2014-12-25
- 2015-12-25
- 2016-12-25

All four dates have documented national Christmas holiday records
(`Navidad`) in the holiday dataset.

Treatment decision:
- Do not automatically replace the missing dates with zero sales.
- Preserve the source data.
- Account for these calendar exceptions when constructing the forecasting
  time series.

### 3. Sales vs Transactions date coverage

Two sales dates have no corresponding transaction records:

- 2016-01-01
- 2016-01-03

2016-01-01 has a documented national holiday record. 2016-01-03 does not
have a corresponding holiday record in the available holiday dataset.

Treatment decision:
- Do not automatically replace missing transaction values with zero.
- Investigate the dates during data preparation.
- Avoid assuming that missing transaction records represent zero activity.

### 4. Holiday date multiplicity

The holiday/event dataset contains multiple records for some dates.

Treatment decision:
- Do not directly join the raw holiday table to sales at Date × Store ×
  Product Family grain.
- Create a controlled date-level holiday/event representation before joining.
- Preserve the ability to distinguish relevant event characteristics.

### 5. Oil price missing observations

The oil dataset contains **43 missing price observations** and therefore
requires treatment if oil is included as a forecasting feature.

Treatment decision:
- Keep the raw values unchanged.
- Evaluate the missing-date pattern during preparation.
- Choose the treatment based on the forecasting methodology and business
  purpose.
- Document the final treatment and its rationale before model evaluation.

### Overall Quality Conclusion

The initial assessment contains:

- **11 PASS**
- **5 REVIEW**
- **0 FAIL**

The source data is therefore considered **suitable to proceed to structured
data preparation**, subject to documenting and appropriately handling the
identified REVIEW items.

In [3]:
# ============================================================
# Step 2B.6 — Create Machine-Readable Data Quality Results
# ============================================================

import pandas as pd

# Create a structured table from the quality checks.
data_quality_results = pd.DataFrame(
    quality_checks,
    columns=["dimension", "check", "status"]
)

# Add a simple numeric flag for easier reporting later.
data_quality_results["status_flag"] = (
    data_quality_results["status"]
    .map({
        "PASS": 1,
        "REVIEW": 0,
        "FAIL": -1
    })
)

# Display the structured results.
display(data_quality_results)

,dimension,check,status,status_flag
0,Completeness,Missing values in sales dataset,PASS,1
1,Completeness,Missing values in supporting datasets,REVIEW,0
2,Validity,Negative sales,PASS,1
3,Validity,Negative promotion counts,PASS,1
4,Validity,Negative transactions,PASS,1
5,Uniqueness,Duplicate sales grain,PASS,1
6,Uniqueness,Duplicate sales IDs,PASS,1
7,Uniqueness,Duplicate transaction grain,PASS,1
8,Referential Integrity,Sales → Store master,PASS,1
9,Referential Integrity,Transactions → Store master,PASS,1


In [4]:
# Verify the structure and status counts.

print("Rows:", len(data_quality_results))
print("Columns:", list(data_quality_results.columns))

print("\nStatus counts:")
print(
    data_quality_results["status"]
    .value_counts()
    .sort_index()
)

print("\nMissing values:")
print(
    data_quality_results.isna().sum()
)

Rows: 16
Columns: ['dimension', 'check', 'status', 'status_flag']

Status counts:
status
PASS      11
REVIEW     5
Name: count, dtype: int64

Missing values:
dimension      0
check          0
status         0
status_flag    0
dtype: int64


In [ ]:
# ============================================================
# Step 2B.9 — Export Data Quality Results
# ============================================================

from pathlib import Path

# Define the validation output path.
validation_dir = Path("../../data/validation")
validation_dir.mkdir(parents=True, exist_ok=True)

quality_results_path = (
    validation_dir / "data_quality_results.csv"
)

# Export the structured quality assessment.
data_quality_results.to_csv(
    quality_results_path,
    index=False
)

print("Exported:", quality_results_path.resolve())
print("Rows exported:", len(data_quality_results))

Exported: D:\GitHub\retail-sales-forecasting\data\validation\data_quality_results.csv
Rows exported: 16


In [6]:
# ============================================================
# Step 2B.10 — Verify Exported Quality Results
# ============================================================

# Read the exported CSV back into Python.
quality_results_check = pd.read_csv(
    quality_results_path
)

print("Rows read back:", len(quality_results_check))
print(
    "Columns:",
    list(quality_results_check.columns)
)

print("\nStatus counts:")
print(
    quality_results_check["status"]
    .value_counts()
    .sort_index()
)

print("\nMissing values:")
print(
    quality_results_check.isna().sum()
)

print("\nFirst 5 rows:")
display(
    quality_results_check.head()
)

Rows read back: 16
Columns: ['dimension', 'check', 'status', 'status_flag']

Status counts:
status
PASS      11
REVIEW     5
Name: count, dtype: int64

Missing values:
dimension      0
check          0
status         0
status_flag    0
dtype: int64

First 5 rows:


,dimension,check,status,status_flag
0,Completeness,Missing values in sales dataset,PASS,1
1,Completeness,Missing values in supporting datasets,REVIEW,0
2,Validity,Negative sales,PASS,1
3,Validity,Negative promotion counts,PASS,1
4,Validity,Negative transactions,PASS,1


# Phase 3 — Data Quality & Preparation

## Step 3.1 — Forecasting Target and Grain

### Forecasting Objective

The initial forecasting objective is to predict **monthly retail sales** at
the total-business level.

### Forecasting Target

The source dataset field `sales` will be used as the forecasting target.

The field will be referred to as **Sales** rather than **Net Sales** because
the source data profile does not establish that `sales` represents net sales.

### Forecasting Grain

The initial forecasting grain is:

**Month × Total Retail Business**

Daily store × product-family sales will therefore be aggregated to monthly
total sales before forecasting.

### Why Monthly Total Sales?

Monthly total sales provides a practical first forecasting layer for:

- Retail management planning
- Sales target setting
- Commercial planning
- Inventory and supply planning
- Budget and business planning

More granular forecasting may be explored later if the data and project
requirements justify it.

### Data Governance Principle

Raw source files will remain unchanged.

All cleaning, transformation, aggregation, and feature engineering will be
performed on working or processed datasets.